<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/02_no_supervisado/22_clustering_jerarquico.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Clustering aglomerativo jerárquico

**Pregunta guía:** ¿Qué estructura multiescala revela un dendrograma?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


Cada observación empieza como un grupo. En cada paso se fusionan los dos
grupos más próximos. `single` usa la pareja mínima, `complete` la máxima,
`average` el promedio y Ward minimiza el aumento de suma de cuadrados.
Ward requiere distancia euclídea. El dendrograma muestra una **secuencia
de fusiones**, no una verdad automática sobre cuántos grupos existen.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
X, y_real = make_blobs(
    n_samples=450,
    centers=[(-3, -1), (0, 2), (2.5, -1), (3.5, 2.8)],
    cluster_std=[0.55, 0.9, 0.6, 0.45],
    random_state=SEMILLA,
)
Xs = StandardScaler().fit_transform(X)

muestra = Xs[np.random.default_rng(SEMILLA).choice(len(Xs), 100, replace=False)]
Z = linkage(muestra, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=25, show_contracted=True)
plt.ylabel("incremento de distancia Ward")
plt.title("Dendrograma truncado")
plt.show()


In [ ]:
filas, modelos = [], {}
for enlace in ["single", "complete", "average", "ward"]:
    for k in range(2, 8):
        modelo = AgglomerativeClustering(n_clusters=k, linkage=enlace)
        etiquetas = modelo.fit_predict(Xs)
        modelos[(enlace, k)] = etiquetas
        filas.append(
            {
                "linkage": enlace,
                "k": k,
                "silhouette": silhouette_score(Xs, etiquetas),
                "ARI_diagnóstico": adjusted_rand_score(y_real, etiquetas),
            }
        )
resultados = pd.DataFrame(filas)
display(resultados.sort_values("silhouette", ascending=False).head(10))

mejor = resultados.sort_values("silhouette").iloc[-1]
etiquetas = modelos[(mejor.linkage, int(mejor.k))]
plt.scatter(*Xs.T, c=etiquetas, s=18, cmap="tab10")
plt.title(f"Mejor silhouette: {mejor.linkage}, k={int(mejor.k)}")
plt.show()


**Ejercicios:** construya a mano las tres primeras fusiones de seis
puntos; compare el efecto cadena de `single`; use distancia coseno con
`average`; aplique bootstrap y mida qué pares de observaciones permanecen
juntos. Esa matriz de coasociación es más informativa que un único corte.
